# 01 — Data Cleaning: FreshMart Perishable Goods

**Business context:** FreshMart's meat department over-orders on slow days and stocks out on peak days. Before analyzing demand patterns, we need a clean, reliable dataset.

**Dataset:** [Kaggle — Perishable Goods Management](https://www.kaggle.com/datasets/likithagedipudi/perishable-goods-management) — 100,000 synthetic transaction records (2023–2024), 50 stores × 5 US regions, 10 product categories.

This notebook:
1. Loads the raw CSV and audits its structure
2. Handles missing values (with reasoning for each choice)
3. Removes duplicates
4. Parses dates and validates ranges
5. Flags outliers for review
6. Saves a cleaned CSV for analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 50)
print('pandas', pd.__version__)

## 1. Load & audit

First, understand what we're working with: shape, columns, dtypes, and a first look at the data.

In [ ]:
# Update the path to your CSV location
df = pd.read_csv('perishable_goods_management.csv')

print('Shape:', df.shape)
print('\nColumns:', list(df.columns))
print('\nDtypes:')
print(df.dtypes)

In [ ]:
df.head()

## 2. Missing values

**Reasoning:** We drop rows only when the missing field is critical and rare. For non-critical fields we keep the row and note the gap, because every transaction represents real demand signal.

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'missing': missing, 'pct': missing_pct})
print(missing_df[missing_df['missing'] > 0].sort_values('missing', ascending=False))
print(f"\nRows with any missing value: {df.isnull().any(axis=1).sum():,} ({df.isnull().any(axis=1).mean()*100:.2f}%)")

## 3. Duplicates

**Reasoning:** Exact duplicate rows are almost certainly data-entry errors, not real repeat transactions (same store, product, timestamp, and quantity). We remove them.

In [ ]:
dups = df.duplicated().sum()
print(f'Duplicate rows: {dups:,} ({dups/len(df)*100:.2f}%)')
df = df.drop_duplicates().reset_index(drop=True)
print('Shape after dedup:', df.shape)

## 4. Date parsing & validation

**Reasoning:** Day-of-week analysis is the core of this project, so dates must parse cleanly. We coerce invalid dates to NaT, report them, and drop only those rows — a transaction without a valid date can't be placed on a weekday.

In [ ]:
# Auto-detect likely date columns
date_cols = [c for c in df.columns if 'date' in c.lower() or 'time' in c.lower()]
print('Candidate date columns:', date_cols)

for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')
    bad = df[col].isna().sum()
    print(f'{col}: {bad:,} unparseable dates')
    if bad > 0:
        print(f'  Range: {df[col].min()} → {df[col].max()}')

## 5. Numeric sanity checks

**Reasoning:** Negative quantities or prices are usually returns or errors. We report them rather than silently dropping — the business needs to know if returns are material.

In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns
print(df[num_cols].describe().T[['count', 'mean', 'std', 'min', 'max']])

for col in num_cols:
    neg = (df[col] < 0).sum()
    if neg > 0:
        print(f'{col}: {neg:,} negative values')

## 6. Save cleaned data

The cleaned file feeds 02_eda and 03_waste_analysis.

In [ ]:
df.to_csv('clean_freshmart_data.csv', index=False)
print(f'Saved clean_freshmart_data.csv — {df.shape[0]:,} rows × {df.shape[1]} columns')